# AI Smart Travel Planning Assistant
### Agentic AI + Agentic RAG — End-to-End Project

This notebook implements an **AI Smart Travel Planning Assistant** that uses:

- **Single Agent** — one Claude agent with a full tool set, deciding for itself which tool to call each turn.
- **Multi-Agent Collaboration** — a `RouterAgent` classifies intent and delegates to specialised agents (RAG Q&A, Itinerary, Comparison, Budget, Weather).
- **Tool Calling** — `get_weather` and `estimate_budget` are exposed to Claude as tools.
- **Agentic RAG** — travel-guide PDFs are chunked, indexed, retrieved, and used to ground generated answers, itineraries and comparisons, with source documents cited.

**LLM brain:** Anthropic Claude, via the official `anthropic` Python SDK.

> **Running without an API key:** if `ANTHROPIC_API_KEY` is not set, the notebook automatically
> falls back to a small deterministic **mock LLM** so every cell below still runs end-to-end and
> you can inspect the full agent control-flow (routing, retrieval, tool calls) with zero cost / zero
> network access. Set the key (next cell) to get real Claude-generated answers.


In [1]:
# 1. Setup — install & imports
# (anthropic, pypdf, scikit-learn, fpdf2 are the only third-party deps)
import os, re, json, textwrap
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Setup OK")


Setup OK


In [2]:
# 2. Configuration
# --------------------------------------------------------------------------
# Put your key here, or (better) set it as an environment variable before
# starting Jupyter:   export ANTHROPIC_API_KEY="sk-ant-..."
#
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."   # <- uncomment & fill in if not using env var

ANTHROPIC_MODEL = os.environ.get("ANTHROPIC_MODEL", "claude-sonnet-4-5-20250929")
API_KEY = os.environ.get("ANTHROPIC_API_KEY")
MOCK_MODE = API_KEY is None

if not MOCK_MODE:
    import anthropic
    _client = anthropic.Anthropic(api_key=API_KEY)

print("Model:", ANTHROPIC_MODEL)
print("MOCK_MODE (no API key found):", MOCK_MODE)


Model: claude-sonnet-4-5-20250929
MOCK_MODE (no API key found): True


## 3. Document Ingestion — loading travel guide PDFs & chunking

Handles: *"Loading travel guide PDFs and destination documents"* and
*"Upload multiple travel guide PDF documents"*.

For a self-contained, runnable demo we first **generate a couple of sample
travel-guide PDFs** on disk (so the notebook needs no external upload to run),
then load them exactly the way real uploaded PDFs would be loaded, using `pypdf`.


In [3]:
# 3a. Generate sample travel-guide PDFs (stand-ins for uploaded documents)
from fpdf import FPDF

SAMPLE_GUIDES = {
    "Goa_Travel_Guide.pdf": '''
    Goa Travel Guide

    Overview: Goa is a coastal state in India famous for its beaches, Portuguese-era
    architecture, and vibrant nightlife.

    Best time to visit: November to February, when the weather is dry and pleasant.
    Avoid the monsoon season (June-September) due to heavy rainfall.

    Top attractions: Baga Beach, Calangute Beach, Palolem Beach, Fort Aguada,
    Basilica of Bom Jesus, Dudhsagar Falls, Anjuna Flea Market.

    Local guidelines: Beach parties must observe quiet hours after 10pm in
    residential zones. Alcohol consumption on beaches after sunset is restricted
    in some areas. Wear modest clothing when visiting churches and temples.

    Budget notes: Backpacker hostels start around $8-12/night; budget travelers
    can manage on roughly $25-30/day including food and local transport.

    Transportation: Scooters and bikes are the most popular way to get around;
    international rentals require a valid license. Buses and taxis are also
    widely available.
    ''',
    "Manali_Travel_Guide.pdf": '''
    Manali Travel Guide

    Overview: Manali is a hill town in Himachal Pradesh, India, popular for
    mountain scenery, adventure sports and snow activities.

    Best time to visit: March to June for pleasant weather and trekking;
    December to February for snowfall and winter sports. Avoid landslide-prone
    monsoon months (July-August) for travel to Rohtang Pass.

    Top attractions: Solang Valley, Rohtang Pass, Hadimba Temple, Old Manali,
    Jogini Falls, Kasol day trip.

    Local guidelines: Rohtang Pass may close without notice due to snow;
    check road conditions before travelling. Carry warm clothing even in
    summer as nights are cold. Permits may be required for some high-altitude
    routes.

    Budget notes: Guesthouses start around $10-15/night; adventure activities
    (paragliding, skiing) can add $20-50/day depending on activity.

    Transportation: Local taxis and shared cabs connect Manali to Solang Valley
    and Rohtang Pass; self-drive rentals are available in town.
    '''
}

os.makedirs("sample_travel_guides", exist_ok=True)
for filename, content in SAMPLE_GUIDES.items():
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Helvetica", size=11)
    for line in textwrap.dedent(content).strip().split("\n"):
        line = line.strip()
        pdf.set_x(pdf.l_margin)
        if line:
            pdf.multi_cell(0, 6, line, new_x="LMARGIN", new_y="NEXT")
        else:
            pdf.ln(4)
    pdf.output(os.path.join("sample_travel_guides", filename))

print("Generated sample PDFs:", os.listdir("sample_travel_guides"))


Generated sample PDFs: ['Manali_Travel_Guide.pdf', 'Goa_Travel_Guide.pdf']


In [4]:
# 3b. PDF loading & text chunking utilities
from pypdf import PdfReader

@dataclass
class Document:
    doc_id: str
    source: str
    text: str

@dataclass
class Chunk:
    chunk_id: str
    source: str
    text: str

def load_pdf_text(path: str) -> str:
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

def chunk_text(text: str, chunk_size: int = 700, overlap: int = 120) -> List[str]:
    text = re.sub(r"\s+", " ", text).strip()
    chunks, start = [], 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return [c.strip() for c in chunks if c.strip()]

def load_documents_from_folder(folder: str) -> List[Document]:
    docs = []
    for i, fname in enumerate(sorted(os.listdir(folder))):
        if fname.lower().endswith(".pdf"):
            path = os.path.join(folder, fname)
            docs.append(Document(doc_id=f"doc{i}", source=fname, text=load_pdf_text(path)))
    return docs

documents = load_documents_from_folder("sample_travel_guides")
for d in documents:
    print(d.source, "->", len(d.text), "characters extracted")


Goa_Travel_Guide.pdf -> 943 characters extracted
Manali_Travel_Guide.pdf -> 946 characters extracted


## 4. Vector Store — embeddings + semantic retrieval (Agentic RAG)

Handles: *"Creating embeddings and storing them in a vector database"* and
*"Retrieving relevant travel information using semantic search"*.

This uses **TF-IDF + cosine similarity** as the embedding/similarity model.
It needs no external model downloads, so the whole pipeline runs fully offline.
In production, swap `VectorStore._reindex` / `search` for real dense
embeddings (e.g. Voyage AI or OpenAI embeddings) stored in Chroma/FAISS/Pinecone —
the rest of the system (agents, RAG, retrieval interface) is unaffected because
they only depend on `add_documents()` / `search()`.


In [5]:
# 4. Vector store implementation
class VectorStore:
    def __init__(self):
        self.chunks: List[Chunk] = []
        self.vectorizer: Optional[TfidfVectorizer] = None
        self.matrix = None

    def add_documents(self, docs: List[Document], chunk_size=700, overlap=120):
        for doc in docs:
            for i, ch in enumerate(chunk_text(doc.text, chunk_size, overlap)):
                self.chunks.append(Chunk(chunk_id=f"{doc.doc_id}-{i}", source=doc.source, text=ch))
        self._reindex()

    def _reindex(self):
        if not self.chunks:
            return
        self.vectorizer = TfidfVectorizer(stop_words="english")
        self.matrix = self.vectorizer.fit_transform([c.text for c in self.chunks])

    def search(self, query: str, k: int = 3) -> List[Dict[str, Any]]:
        if not self.chunks or self.vectorizer is None:
            return []
        qv = self.vectorizer.transform([query])
        sims = cosine_similarity(qv, self.matrix)[0]
        top_idx = np.argsort(sims)[::-1][:k]
        return [{"source": self.chunks[i].source, "text": self.chunks[i].text, "score": float(sims[i])}
                for i in top_idx if sims[i] > 0]

vector_store = VectorStore()
vector_store.add_documents(documents)
print(f"Indexed {len(vector_store.chunks)} chunks from {len(documents)} documents")

# quick sanity check
for r in vector_store.search("best time to visit for snow", k=2):
    print(f"  [{r['score']:.3f}] {r['source']} -> {r['text'][:90]}...")


Indexed 4 chunks from 2 documents
  [0.261] Manali_Travel_Guide.pdf -> Manali Travel Guide Overview: Manali is a hill town in Himachal Pradesh, India, popular fo...
  [0.123] Goa_Travel_Guide.pdf -> Goa Travel Guide Overview: Goa is a coastal state in India famous for its beaches, Portugu...


## 5. Tools — weather & budget (Tool Calling)

Handles: *"Providing weather ... information through tool calling"* and
*"Estimating trip budgets"*. These are plain Python functions plus a Claude
`tools` JSON-schema so the LLM can decide *when* and *how* to call them.


In [6]:
# 5. Tool implementations + schemas
WEATHER_DB = {
    "goa": {"forecast": "Sunny, light coastal breeze", "temp_c": 31, "advisory": "High humidity, carry sunscreen"},
    "manali": {"forecast": "Snow showers", "temp_c": -2, "advisory": "Roads may be icy, carry warm layers"},
    "jaipur": {"forecast": "Clear skies", "temp_c": 27, "advisory": "Dry heat, stay hydrated"},
}

COST_PER_DAY_USD = {
    "budget": {"stay": 15, "food": 10, "local_transport": 5},
    "mid-range": {"stay": 45, "food": 25, "local_transport": 12},
    "luxury": {"stay": 150, "food": 60, "local_transport": 30},
}

def tool_get_weather(city: str) -> Dict[str, Any]:
    # TOOL: (simulated) current weather + advisory. Swap for a real weather
    # API (e.g. OpenWeatherMap) in production.
    data = WEATHER_DB.get(city.strip().lower(),
                           {"forecast": "Data unavailable", "temp_c": None, "advisory": "No advisory available"})
    return {"city": city, **data}

def tool_estimate_budget(destination: str, days: int, style: str = "mid-range", travelers: int = 1) -> Dict[str, Any]:
    # TOOL: estimates a trip budget from a simple per-day cost model.
    rates = COST_PER_DAY_USD.get(style.lower(), COST_PER_DAY_USD["mid-range"])
    total = sum(rates.values()) * days * travelers
    return {"destination": destination, "days": days, "style": style, "travelers": travelers,
            "breakdown_per_day_per_person_usd": rates, "total_estimated_usd": total}

TOOL_SCHEMAS = [
    {"name": "get_weather", "description": "Get current weather forecast and travel advisory for a city.",
     "input_schema": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}},
    {"name": "estimate_budget", "description": "Estimate total trip budget in USD for a destination.",
     "input_schema": {"type": "object", "properties": {
         "destination": {"type": "string"}, "days": {"type": "integer"},
         "style": {"type": "string", "enum": ["budget", "mid-range", "luxury"]},
         "travelers": {"type": "integer"}}, "required": ["destination", "days"]}},
]
TOOL_IMPL = {"get_weather": tool_get_weather, "estimate_budget": tool_estimate_budget}

print(tool_get_weather("Manali"))
print(tool_estimate_budget("Goa", 4, "budget"))


{'city': 'Manali', 'forecast': 'Snow showers', 'temp_c': -2, 'advisory': 'Roads may be icy, carry warm layers'}
{'destination': 'Goa', 'days': 4, 'style': 'budget', 'travelers': 1, 'breakdown_per_day_per_person_usd': {'stay': 15, 'food': 10, 'local_transport': 5}, 'total_estimated_usd': 120}


## 6. LLM Client — Claude wrapper (real API, with offline mock fallback)

All agents below call `llm_call(...)` only — never the SDK directly — so the
same control flow (including the `tool_use` loop) is exercised whether or not
a real API key is configured.


In [7]:
# 6. LLM wrapper
class _MockBlock:
    def __init__(self, type_, **kw):
        self.type = type_
        for k, v in kw.items():
            setattr(self, k, v)

class _MockResponse:
    def __init__(self, content, stop_reason="end_turn"):
        self.content = content
        self.stop_reason = stop_reason

def _mock_llm_response(system, messages, tools):
    # Deterministic stand-in for Claude, used only when ANTHROPIC_API_KEY is unset.
    # Mirrors the real API's tool_use / text response shapes so agent code is identical either way.
    last_user = ""
    for m in reversed(messages):
        if m["role"] == "user":
            last_user = m["content"] if isinstance(m["content"], str) else json.dumps(m["content"])
            break
    lower = last_user.lower()

    if tools:
        if "weather" in lower:
            city = next((c for c in WEATHER_DB if c in lower), "goa")
            return _MockResponse([_MockBlock("tool_use", id="mock_w", name="get_weather", input={"city": city})],
                                  stop_reason="tool_use")
        if "budget" in lower or "cost" in lower:
            return _MockResponse([_MockBlock("tool_use", id="mock_b", name="estimate_budget",
                                              input={"destination": "Goa", "days": 5, "style": "mid-range", "travelers": 1})],
                                  stop_reason="tool_use")

    text = (f"[MOCK RESPONSE - no ANTHROPIC_API_KEY set]\n"
            f"System role: {system[:70]}...\n"
            f"Responding to: {last_user[:150]}\n"
            f"(Set ANTHROPIC_API_KEY to get real Claude-generated answers.)")
    return _MockResponse([_MockBlock("text", text=text)])

def llm_call(system: str, messages: List[Dict[str, Any]], tools: Optional[List[Dict]] = None, max_tokens: int = 1000):
    if MOCK_MODE:
        return _mock_llm_response(system, messages, tools)
    kwargs = dict(model=ANTHROPIC_MODEL, max_tokens=max_tokens, system=system, messages=messages)
    if tools:
        kwargs["tools"] = tools
    return _client.messages.create(**kwargs)

# smoke test
r = llm_call("You are a helpful travel agent.", [{"role": "user", "content": "weather in manali?"}], tools=TOOL_SCHEMAS)
print(r.stop_reason, r.content[0].type)


tool_use tool_use


## 7. Specialised Agents

- `RAGAgent` — Agentic RAG Q&A with cited sources.
- `ItineraryAgent` — RAG-grounded day-by-day itinerary generation.
- `ComparisonAgent` — compares multiple destinations using retrieved context.
- `WeatherAgent`, `BudgetAgent` — thin wrappers around the tools above.


In [8]:
# 7. Agents
class RAGAgent:
    # Agentic RAG: retrieve -> generate -> cite sources.
    def __init__(self, vector_store: VectorStore):
        self.vs = vector_store

    def answer(self, query: str, k: int = 3) -> Dict[str, Any]:
        hits = self.vs.search(query, k=k)
        context = "\n\n".join(f"[Source: {h['source']}]\n{h['text']}" for h in hits) or "No relevant context found."
        system = ("You are a travel research assistant. Answer ONLY using the provided context. "
                   "If the context lacks the answer, say so. Always name the source document(s) used.")
        resp = llm_call(system, [{"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}])
        text = "".join(b.text for b in resp.content if b.type == "text")
        return {"answer": text, "sources": sorted(set(h["source"] for h in hits))}


class ItineraryAgent:
    def __init__(self, vector_store: VectorStore):
        self.vs = vector_store

    def generate(self, destination: str, days: int, interests: str = "") -> Dict[str, Any]:
        hits = self.vs.search(f"{destination} attractions things to do {interests}", k=4)
        context = "\n\n".join(h["text"] for h in hits)
        system = "You are an expert travel-itinerary planner. Produce a concise day-by-day plan."
        prompt = (f"Destination: {destination}\nDuration: {days} days\n"
                  f"Interests: {interests or 'general sightseeing'}\nContext:\n{context}\n\n"
                  f"Write a {days}-day itinerary, one short paragraph per day.")
        resp = llm_call(system, [{"role": "user", "content": prompt}])
        text = "".join(b.text for b in resp.content if b.type == "text")
        return {"itinerary": text, "sources": sorted(set(h["source"] for h in hits))}


class ComparisonAgent:
    def __init__(self, vector_store: VectorStore):
        self.vs = vector_store

    def compare(self, destinations: List[str]) -> Dict[str, Any]:
        hits = []
        for d in destinations:
            hits += self.vs.search(d, k=3)
        context = "\n\n".join(f"[{h['source']}] {h['text']}" for h in hits)
        system = "You are a travel advisor comparing destinations for a traveler."
        prompt = (f"Compare these destinations: {', '.join(destinations)}.\n\nContext:\n{context}\n\n"
                  "Give a short comparison (best time to visit, vibe, key attractions) and a 1-line recommendation.")
        resp = llm_call(system, [{"role": "user", "content": prompt}])
        text = "".join(b.text for b in resp.content if b.type == "text")
        return {"comparison": text, "sources": sorted(set(h["source"] for h in hits))}


class WeatherAgent:
    def run(self, city: str) -> Dict[str, Any]:
        return tool_get_weather(city)


class BudgetAgent:
    def run(self, destination: str, days: int, style: str = "mid-range", travelers: int = 1) -> Dict[str, Any]:
        return tool_estimate_budget(destination, days, style, travelers)

print("Agents defined: RAGAgent, ItineraryAgent, ComparisonAgent, WeatherAgent, BudgetAgent")


Agents defined: RAGAgent, ItineraryAgent, ComparisonAgent, WeatherAgent, BudgetAgent


## 8. Router Agent — Multi-Agent Collaboration pattern

The `RouterAgent` classifies the user's natural-language request into one of
five intents (this is the required *"Agent Workflow to determine the
appropriate action"*) and delegates to the matching specialised agent above.


In [9]:
# 8. RouterAgent
class RouterAgent:
    INTENTS = ["rag_qa", "itinerary", "compare", "budget", "weather"]

    def __init__(self, vector_store: VectorStore):
        self.rag_agent = RAGAgent(vector_store)
        self.itinerary_agent = ItineraryAgent(vector_store)
        self.comparison_agent = ComparisonAgent(vector_store)
        self.budget_agent = BudgetAgent()
        self.weather_agent = WeatherAgent()

    def classify(self, query: str) -> str:
        if MOCK_MODE:
            return self._fallback_classify(query)
        system = f"Classify the travel query into exactly one label: {', '.join(self.INTENTS)}. Reply with only the label."
        resp = llm_call(system, [{"role": "user", "content": query}])
        label = "".join(b.text for b in resp.content if b.type == "text").strip().lower()
        for intent in self.INTENTS:
            if intent in label:
                return intent
        return self._fallback_classify(query)

    @staticmethod
    def _fallback_classify(query: str) -> str:
        q = query.lower()
        if "weather" in q: return "weather"
        if "budget" in q or "cost" in q or "how much" in q: return "budget"
        if "compare" in q or " vs " in q: return "compare"
        if "itinerary" in q or "plan" in q or "days" in q: return "itinerary"
        return "rag_qa"

    def handle(self, query: str, **kw) -> Dict[str, Any]:
        intent = self.classify(query)
        if intent == "weather":
            return {"intent": intent, "result": self.weather_agent.run(kw.get("city", "Goa"))}
        if intent == "budget":
            return {"intent": intent, "result": self.budget_agent.run(
                kw.get("destination", "Goa"), kw.get("days", 5), kw.get("style", "mid-range"), kw.get("travelers", 1))}
        if intent == "compare":
            return {"intent": intent, "result": self.comparison_agent.compare(kw.get("destinations", ["Goa", "Manali"]))}
        if intent == "itinerary":
            return {"intent": intent, "result": self.itinerary_agent.generate(
                kw.get("destination", "Goa"), kw.get("days", 3), kw.get("interests", ""))}
        return {"intent": intent, "result": self.rag_agent.answer(query)}

router = RouterAgent(vector_store)
print("RouterAgent ready")


RouterAgent ready


In [10]:
# 8b. Demo: RouterAgent handling each of the 5 intents
print("--- RAG Q&A ---")
res = router.handle("What is the best time to visit Goa?")
print("Intent:", res["intent"])
print(res["result"]["answer"])
print("Sources:", res["result"]["sources"])

print("\n--- Itinerary ---")
res = router.handle("Plan a trip", destination="Manali", days=3, interests="adventure sports")
print("Intent:", res["intent"])
print(res["result"]["itinerary"])
print("Sources:", res["result"]["sources"])


--- RAG Q&A ---
Intent: rag_qa
[MOCK RESPONSE - no ANTHROPIC_API_KEY set]
System role: You are a travel research assistant. Answer ONLY using the provided co...
Responding to: Context:
[Source: Goa_Travel_Guide.pdf]
Goa Travel Guide Overview: Goa is a coastal state in India famous for its beaches, Portuguese-era architecture
(Set ANTHROPIC_API_KEY to get real Claude-generated answers.)
Sources: ['Goa_Travel_Guide.pdf', 'Manali_Travel_Guide.pdf']

--- Itinerary ---
Intent: itinerary
[MOCK RESPONSE - no ANTHROPIC_API_KEY set]
System role: You are an expert travel-itinerary planner. Produce a concise day-by-d...
Responding to: Destination: Manali
Duration: 3 days
Interests: adventure sports
Context:
Manali Travel Guide Overview: Manali is a hill town in Himachal Pradesh, Ind
(Set ANTHROPIC_API_KEY to get real Claude-generated answers.)
Sources: ['Goa_Travel_Guide.pdf', 'Manali_Travel_Guide.pdf']


In [11]:
print("--- Comparison ---")
res = router.handle("Compare Goa vs Manali", destinations=["Goa", "Manali"])
print("Intent:", res["intent"])
print(res["result"]["comparison"])
print("Sources:", res["result"]["sources"])

print("\n--- Budget ---")
res = router.handle("How much would a 4 day budget trip to Goa cost?", destination="Goa", days=4, style="budget")
print("Intent:", res["intent"])
print(res["result"])

print("\n--- Weather ---")
res = router.handle("What's the weather in Manali right now?", city="manali")
print("Intent:", res["intent"])
print(res["result"])


--- Comparison ---
Intent: compare
[MOCK RESPONSE - no ANTHROPIC_API_KEY set]
System role: You are a travel advisor comparing destinations for a traveler....
Responding to: Compare these destinations: Goa, Manali.

Context:
[Goa_Travel_Guide.pdf] Goa Travel Guide Overview: Goa is a coastal state in India famous for its be
(Set ANTHROPIC_API_KEY to get real Claude-generated answers.)
Sources: ['Goa_Travel_Guide.pdf', 'Manali_Travel_Guide.pdf']

--- Budget ---
Intent: budget
{'destination': 'Goa', 'days': 4, 'style': 'budget', 'travelers': 1, 'breakdown_per_day_per_person_usd': {'stay': 15, 'food': 10, 'local_transport': 5}, 'total_estimated_usd': 120}

--- Weather ---
Intent: weather
{'city': 'manali', 'forecast': 'Snow showers', 'temp_c': -2, 'advisory': 'Roads may be icy, carry warm layers'}


## 9. Single Agent — one agent, full tool set, self-directed (Agentic AI loop)

Contrast with `RouterAgent` above: here there is **one** Claude agent holding
*all* tools (`get_weather`, `estimate_budget`, `search_knowledge_base`). It
runs Claude's standard `tool_use` loop, deciding for itself, turn by turn,
which tool (if any) it needs before producing a final answer.


In [12]:
# 9. SingleAgent
class SingleAgent:
    def __init__(self, vector_store: VectorStore):
        self.vs = vector_store
        self.tools = TOOL_SCHEMAS + [{
            "name": "search_knowledge_base",
            "description": "Semantic search over uploaded travel-guide documents.",
            "input_schema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
        }]
        self.system = ("You are an AI Smart Travel Planning Assistant. You have tools for weather, "
                       "budget estimation and knowledge-base search. Use tools when needed, cite sources "
                       "from the knowledge base when you use it, and give a clear final answer.")

    def _run_tool(self, name: str, inp: Dict[str, Any]) -> Any:
        if name == "search_knowledge_base":
            return [{"source": h["source"], "text": h["text"]} for h in self.vs.search(inp["query"], k=3)]
        return TOOL_IMPL[name](**inp)

    def chat(self, user_query: str, max_turns: int = 4) -> Dict[str, Any]:
        messages = [{"role": "user", "content": user_query}]
        trace = []
        for _ in range(max_turns):
            resp = llm_call(self.system, messages, tools=self.tools)
            if resp.stop_reason == "tool_use":
                tb = next(b for b in resp.content if b.type == "tool_use")
                result = self._run_tool(tb.name, tb.input)
                trace.append({"tool": tb.name, "input": tb.input, "output": result})
                messages.append({"role": "assistant", "content": [
                    {"type": "tool_use", "id": tb.id, "name": tb.name, "input": tb.input}]})
                messages.append({"role": "user", "content": [
                    {"type": "tool_result", "tool_use_id": tb.id, "content": str(result)}]})
                continue
            text = "".join(b.text for b in resp.content if b.type == "text")
            return {"answer": text, "trace": trace}
        return {"answer": "(max turns reached)", "trace": trace}

single_agent = SingleAgent(vector_store)
result = single_agent.chat("What's the weather in Goa and what should I pack?")
print("Answer:\n", result["answer"])
print("\nTool trace:", result["trace"])


Answer:
 [MOCK RESPONSE - no ANTHROPIC_API_KEY set]
System role: You are an AI Smart Travel Planning Assistant. You have tools for weat...
Responding to: [{"type": "tool_result", "tool_use_id": "mock_w", "content": "{'city': 'goa', 'forecast': 'Sunny, light coastal breeze', 'temp_c': 31, 'advisory': 'Hi
(Set ANTHROPIC_API_KEY to get real Claude-generated answers.)

Tool trace: [{'tool': 'get_weather', 'input': {'city': 'goa'}, 'output': {'city': 'goa', 'forecast': 'Sunny, light coastal breeze', 'temp_c': 31, 'advisory': 'High humidity, carry sunscreen'}}]


## 10. Simple chat interface

A minimal REPL wrapper around `RouterAgent` for interactive use. Run the cell
below and type queries (type `exit` to stop) — or call `chat_once(query, **kwargs)`
programmatically as shown.


In [13]:
# 10. Interactive helper
def chat_once(query: str, **kwargs) -> None:
    res = router.handle(query, **kwargs)
    print(f"[intent: {res['intent']}]")
    result = res["result"]
    if isinstance(result, dict) and "answer" in result:
        print(result["answer"]); print("Sources:", result.get("sources"))
    elif isinstance(result, dict) and "itinerary" in result:
        print(result["itinerary"]); print("Sources:", result.get("sources"))
    elif isinstance(result, dict) and "comparison" in result:
        print(result["comparison"]); print("Sources:", result.get("sources"))
    else:
        print(json.dumps(result, indent=2))

def chat_repl():
    # Run this in a real (non-headless) Jupyter session for an interactive loop.
    print("Smart Travel Assistant — type 'exit' to quit")
    while True:
        q = input("You: ")
        if q.strip().lower() in {"exit", "quit"}:
            break
        chat_once(q)

# Example programmatic call (safe to run non-interactively):
chat_once("Tell me about attractions in Goa")


[intent: rag_qa]
[MOCK RESPONSE - no ANTHROPIC_API_KEY set]
System role: You are a travel research assistant. Answer ONLY using the provided co...
Responding to: Context:
[Source: Goa_Travel_Guide.pdf]
Goa Travel Guide Overview: Goa is a coastal state in India famous for its beaches, Portuguese-era architecture
(Set ANTHROPIC_API_KEY to get real Claude-generated answers.)
Sources: ['Goa_Travel_Guide.pdf', 'Manali_Travel_Guide.pdf']


## 11. Summary — deliverables mapping

| Brief requirement | Implemented as |
|---|---|
| Understand NL travel requirements | `RouterAgent.classify`, `SingleAgent.chat` |
| Load travel guide PDFs | Section 3 (`load_documents_from_folder`, `pypdf`) |
| Create embeddings & vector DB | Section 4 (`VectorStore`, TF-IDF + cosine similarity) |
| Retrieve via semantic search | `VectorStore.search` |
| Agent workflow decides action | `RouterAgent.handle` / `SingleAgent` tool_use loop |
| Generate itineraries | `ItineraryAgent` |
| Compare destinations | `ComparisonAgent` |
| Estimate budgets | `BudgetAgent` / `tool_estimate_budget` |
| Weather via tool calling | `WeatherAgent` / `tool_get_weather` |
| Agentic RAG Q&A with sources | `RAGAgent` |

**To use real Claude output:** set `ANTHROPIC_API_KEY` in Section 2 and re-run
the notebook — every cell above is unchanged; only `MOCK_MODE` flips to `False`
and `llm_call` starts hitting the real Anthropic API.
